# Pretrain Baseline Model Comparison


In [ ]:

import json
import logging
import random
import time
import traceback
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
)
from torch.utils.data import DataLoader, Dataset
import os

sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', message='enable_nested_tensor is True.*')
warnings.filterwarnings('ignore', category=DeprecationWarning)

PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')


def env_int(name, default):
    value = os.environ.get(name, '').strip()
    if value in {'', '0'}:
        return default
    try:
        parsed = int(value)
    except ValueError:
        return default
    return parsed if parsed > 0 else default
MAIN_LABELS = ['Normal', 'Anterior', 'Inferior', 'Lateral']
MAIN_LABEL_DISPLAY = ['NORM', 'AMI', 'IMI', 'Lateral-involved MI']
MAIN_LABEL_TO_INDEX = {label: idx for idx, label in enumerate(MAIN_LABELS)}
LEAD_ORDER = ['I', 'aVL', 'II', 'III', 'aVF', 'aVR', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

class PerLeadNormalizer:
    def __init__(self, eps=1e-6):
        self.eps=eps; self.mean=None; self.std=None
    def fit(self,x):
        self.mean=x.mean(axis=(0,1),keepdims=True); self.std=np.maximum(x.std(axis=(0,1),keepdims=True),self.eps); return self
    def transform(self,x): return ((x-self.mean)/self.std).astype(np.float32)
    def save(self,path):
        np.save(Path(path)/'normalizer_mean.npy', self.mean.astype(np.float32)); np.save(Path(path)/'normalizer_std.npy', self.std.astype(np.float32))
    def load(self,path):
        self.mean=np.load(Path(path)/'normalizer_mean.npy'); self.std=np.load(Path(path)/'normalizer_std.npy'); return self

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def parse_labels(labels_df, label_arr):
    if 'main_label_name' in labels_df.columns:
        return labels_df['main_label_name'].map(MAIN_LABEL_TO_INDEX).to_numpy(np.int64)
    if label_arr.ndim > 1:
        return label_arr.argmax(axis=1).astype(np.int64)
    return label_arr.reshape(-1).astype(np.int64)

def load_split(dataset_dir, split):
    split_dir=Path(dataset_dir)/split
    x=np.load(split_dir/'x_beats.npy').astype(np.float32)
    label_arr=np.load(split_dir/'main_label.npy')
    meta=pd.read_csv(split_dir/'metadata.csv')
    y=parse_labels(meta,label_arr)
    return x,y,meta

class ECGDataset(Dataset):
    def __init__(self,x,y,indices=None):
        self.x=torch.tensor(x,dtype=torch.float32); self.y=torch.tensor(y,dtype=torch.long)
        self.indices=np.arange(len(y)) if indices is None else np.asarray(indices)
    def __len__(self): return len(self.y)
    def __getitem__(self,idx): return {'x':self.x[idx], 'y':self.y[idx], 'idx':torch.tensor(int(self.indices[idx]),dtype=torch.long)}

def softmax_np(logits):
    logits=logits-logits.max(axis=1,keepdims=True); exp=np.exp(logits); return exp/exp.sum(axis=1,keepdims=True)

def compute_metrics(y_true, logits):
    prob=softmax_np(logits); pred=prob.argmax(axis=1); cm=confusion_matrix(y_true,pred,labels=np.arange(len(MAIN_LABELS)))
    out={'accuracy':float(accuracy_score(y_true,pred)),'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),'macro_f1':float(f1_score(y_true,pred,average='macro',zero_division=0)),'micro_f1':float(f1_score(y_true,pred,average='micro',zero_division=0)),'weighted_f1':float(f1_score(y_true,pred,average='weighted',zero_division=0))}
    per=f1_score(y_true,pred,labels=np.arange(len(MAIN_LABELS)),average=None,zero_division=0); class_rows=[]; aucs=[]
    for idx,(label,display,val) in enumerate(zip(MAIN_LABELS,MAIN_LABEL_DISPLAY,per)):
        tp=float(cm[idx,idx]); fn=float(cm[idx,:].sum()-cm[idx,idx]); fp=float(cm[:,idx].sum()-cm[idx,idx]); tn=float(cm.sum()-tp-fn-fp)
        sens=tp/max(tp+fn,1.0); spec=tn/max(tn+fp,1.0)
        try: auc=float(roc_auc_score((y_true==idx).astype(int),prob[:,idx])); aucs.append(auc)
        except ValueError: auc=np.nan
        out[f'f1_{display}']=float(val); out[f'sensitivity_{display}']=float(sens); out[f'specificity_{display}']=float(spec); out[f'auc_{display}']=auc
        class_rows.append({'label':display,'internal_label':label,'f1_score':float(val),'sensitivity':float(sens),'specificity':float(spec),'auc':auc,'support':int((y_true==idx).sum())})
    out['macro_auc']=float(np.nanmean(aucs)) if aucs else np.nan
    return out, prob, pred, pd.DataFrame(class_rows)

def save_curves(y_true, prob, metrics_dir, prefix):
    metrics_dir=Path(metrics_dir); metrics_dir.mkdir(parents=True,exist_ok=True)
    roc_rows=[]; fig,ax=plt.subplots(figsize=(9,7),dpi=400)
    for idx,label in enumerate(MAIN_LABEL_DISPLAY):
        binary=(y_true==idx).astype(int)
        if binary.min()==binary.max(): continue
        fpr,tpr,_=roc_curve(binary,prob[:,idx]); auc=roc_auc_score(binary,prob[:,idx])
        ax.plot(fpr,tpr,linewidth=2.2,label=f'{label} AUC={auc:.3f}')
        roc_rows.extend([{'label':label,'fpr':float(x),'tpr':float(y),'auc':float(auc)} for x,y in zip(fpr,tpr)])
    ax.plot([0,1],[0,1],'--',color='black'); ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate / Sensitivity'); ax.legend(); ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(metrics_dir/f'{prefix}_roc_curve.png',bbox_inches='tight'); plt.close(fig)
    pd.DataFrame(roc_rows).to_csv(metrics_dir/f'{prefix}_roc_curve.csv',index=False)
    pr_rows=[]; fig,ax=plt.subplots(figsize=(9,7),dpi=400)
    for idx,label in enumerate(MAIN_LABEL_DISPLAY):
        binary=(y_true==idx).astype(int)
        if binary.min()==binary.max(): continue
        precision,recall,_=precision_recall_curve(binary,prob[:,idx]); auprc=average_precision_score(binary,prob[:,idx])
        ax.plot(recall,precision,linewidth=2.2,label=f'{label} AUPRC={auprc:.3f}')
        pr_rows.extend([{'label':label,'recall':float(r),'precision':float(p),'auprc':float(auprc)} for r,p in zip(recall,precision)])
    ax.set_xlabel('Recall / Sensitivity'); ax.set_ylabel('Precision'); ax.legend(); ax.grid(True,alpha=.25); fig.tight_layout(); fig.savefig(metrics_dir/f'{prefix}_pr_curve.png',bbox_inches='tight'); plt.close(fig)
    pd.DataFrame(pr_rows).to_csv(metrics_dir/f'{prefix}_pr_curve.csv',index=False)

def save_confusion(y_true,pred,metrics_dir,prefix):
    cm=confusion_matrix(y_true,pred,labels=np.arange(len(MAIN_LABELS))); metrics_dir=Path(metrics_dir)
    pd.DataFrame(cm,index=MAIN_LABEL_DISPLAY,columns=MAIN_LABEL_DISPLAY).to_csv(metrics_dir/f'{prefix}_confusion_matrix.csv')
    fig,ax=plt.subplots(figsize=(9,8),dpi=400); sns.heatmap(cm,annot=True,fmt='d',cmap='Oranges',xticklabels=MAIN_LABEL_DISPLAY,yticklabels=MAIN_LABEL_DISPLAY,linewidths=1,linecolor='black',ax=ax,annot_kws={'fontsize':14})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); fig.tight_layout(); fig.savefig(metrics_dir/f'{prefix}_confusion_matrix.png',bbox_inches='tight'); plt.close(fig)

def plot_training(rows,metrics_dir):
    df=pd.DataFrame(rows); df.to_csv(Path(metrics_dir)/'metrics.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(14,5),dpi=300)
    axes[0].plot(df['epoch'],df['train_loss'],label='train'); axes[0].plot(df['epoch'],df['val_loss'],label='val'); axes[0].legend(); axes[0].grid(True,alpha=.3); axes[0].set_title('Loss')
    axes[1].plot(df['epoch'],df['train_macro_f1'],label='train'); axes[1].plot(df['epoch'],df['val_macro_f1'],label='val'); axes[1].legend(); axes[1].grid(True,alpha=.3); axes[1].set_title('Macro F1')
    fig.tight_layout(); fig.savefig(Path(metrics_dir)/'training_curve.png',bbox_inches='tight'); plt.close(fig)

def run_epoch(model,loader,criterion,optimizer=None,device='cpu',max_batches=None):
    training=optimizer is not None; model.train(training); total=0; n=0; logits=[]; y=[]; idx=[]
    ctx=torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for bi,b in enumerate(loader):
            if max_batches and bi>=max_batches: break
            xb=b['x'].to(device); yb=b['y'].to(device)
            if training: optimizer.zero_grad(set_to_none=True)
            out=model(xb); loss=criterion(out,yb)
            if training:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            total += float(loss.detach().cpu())*xb.size(0); n += xb.size(0)
            logits.append(out.detach().cpu().numpy()); y.append(yb.detach().cpu().numpy()); idx.append(b['idx'].detach().cpu().numpy())
    return {'loss':total/max(n,1),'logits':np.concatenate(logits),'y_true':np.concatenate(y),'idx':np.concatenate(idx)}

class CNN1D(nn.Module):
    def __init__(self,input_leads=12,num_classes=4):
        super().__init__(); self.net=nn.Sequential(nn.Conv1d(input_leads,64,7,padding=3),nn.BatchNorm1d(64),nn.GELU(),nn.MaxPool1d(2),nn.Conv1d(64,128,5,padding=2),nn.BatchNorm1d(128),nn.GELU(),nn.AdaptiveAvgPool1d(1)); self.head=nn.Linear(128,num_classes)
    def forward(self,x): x=x.permute(0,2,1); return self.head(self.net(x).squeeze(-1))
class LSTMModel(nn.Module):
    def __init__(self,input_leads=12,num_classes=4,hidden=96):
        super().__init__(); self.rnn=nn.LSTM(input_leads,hidden,batch_first=True,bidirectional=True); self.head=nn.Sequential(nn.LayerNorm(hidden*2),nn.Linear(hidden*2,num_classes))
    def forward(self,x): out,_=self.rnn(x); return self.head(out[:,-1])
class GRUModel(nn.Module):
    def __init__(self,input_leads=12,num_classes=4,hidden=96):
        super().__init__(); self.rnn=nn.GRU(input_leads,hidden,batch_first=True,bidirectional=True); self.head=nn.Sequential(nn.LayerNorm(hidden*2),nn.Linear(hidden*2,num_classes))
    def forward(self,x): out,_=self.rnn(x); return self.head(out[:,-1])
class CNNLSTM(nn.Module):
    def __init__(self,input_leads=12,num_classes=4,hidden=96):
        super().__init__(); self.cnn=nn.Sequential(nn.Conv1d(input_leads,64,5,padding=2),nn.BatchNorm1d(64),nn.GELU(),nn.Conv1d(64,64,3,padding=1),nn.BatchNorm1d(64),nn.GELU()); self.rnn=nn.LSTM(64,hidden,batch_first=True,bidirectional=True); self.head=nn.Sequential(nn.LayerNorm(hidden*2),nn.Linear(hidden*2,num_classes))
    def forward(self,x): z=self.cnn(x.permute(0,2,1)).permute(0,2,1); out,_=self.rnn(z); return self.head(out[:,-1])

def build_model(name):
    if name=='cnn1d': return CNN1D()
    if name=='lstm': return LSTMModel()
    if name=='gru': return GRUModel()
    if name=='cnn_lstm': return CNNLSTM()
    raise ValueError(name)

CONFIG={'seed':env_int('CLICNET_SEED', 42),'dataset_dir':str(PROJECT_ROOT/'dataset'/'ptb_xl'),'output_base_dir':str(PROJECT_ROOT/'outputs'/'8_pretrain_model_comparison'),'batch_size':64,'epochs':env_int('CLICNET_EPOCHS', 50),'learning_rate':1e-4,'weight_decay':1e-4,'device':'cuda' if torch.cuda.is_available() else 'cpu','models':['cnn1d','lstm','gru','cnn_lstm']}
set_seed(CONFIG['seed']); OUTPUT_DIR=Path(CONFIG['output_base_dir'])/RUN_TIMESTAMP; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
with open(OUTPUT_DIR/'config.json','w') as f: json.dump(CONFIG,f,indent=2)

x_train,y_train,meta_train=load_split(CONFIG['dataset_dir'],'train'); x_val,y_val,meta_val=load_split(CONFIG['dataset_dir'],'val'); x_test,y_test,meta_test=load_split(CONFIG['dataset_dir'],'test')
normalizer=PerLeadNormalizer().fit(x_train); x_train=normalizer.transform(x_train); x_val=normalizer.transform(x_val); x_test=normalizer.transform(x_test)
train_loader=DataLoader(ECGDataset(x_train,y_train),batch_size=CONFIG['batch_size'],shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available())
val_loader=DataLoader(ECGDataset(x_val,y_val),batch_size=CONFIG['batch_size'],shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(ECGDataset(x_test,y_test),batch_size=CONFIG['batch_size'],shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available())

summary=[]
for model_name in CONFIG['models']:
    model_dir=OUTPUT_DIR/model_name; metrics_dir=model_dir/'metrics'; ckpt_dir=model_dir/'checkpoints'; pred_dir=model_dir/'predictions'; transfer_dir=model_dir/'transfer_ready'
    for d in [metrics_dir,ckpt_dir,pred_dir,transfer_dir,model_dir/'configs']: d.mkdir(parents=True,exist_ok=True)
    with open(model_dir/'configs'/'config.json','w') as f: json.dump({**CONFIG,'model_name':model_name},f,indent=2)
    normalizer.save(transfer_dir)
    model=build_model(model_name).to(CONFIG['device']); criterion=nn.CrossEntropyLoss(); optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG['learning_rate'],weight_decay=CONFIG['weight_decay']); scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=.5,patience=5)
    best=-np.inf; rows=[]
    for epoch in range(1,CONFIG['epochs']+1):
        tr=run_epoch(model,train_loader,criterion,optimizer,CONFIG['device']); va=run_epoch(model,val_loader,criterion,None,CONFIG['device'])
        tm,_,_,_=compute_metrics(tr['y_true'],tr['logits']); vm,_,_,_=compute_metrics(va['y_true'],va['logits']); scheduler.step(vm['macro_f1'])
        rows.append({'epoch':epoch,'train_loss':tr['loss'],'val_loss':va['loss'],'train_macro_f1':tm['macro_f1'],'val_macro_f1':vm['macro_f1'],'learning_rate':optimizer.param_groups[0]['lr']})
        torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'config':CONFIG,'model_name':model_name},ckpt_dir/'last.pt')
        if vm['macro_f1']>best:
            best=vm['macro_f1']; torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'best_macro_f1':best,'config':CONFIG,'model_name':model_name},ckpt_dir/'best_macro_f1.pt')
    plot_training(rows,metrics_dir)
    payload=torch.load(ckpt_dir/'best_macro_f1.pt',map_location=CONFIG['device']); model.load_state_dict(payload['model_state_dict'])
    val=run_epoch(model,val_loader,criterion,None,CONFIG['device']); test=run_epoch(model,test_loader,criterion,None,CONFIG['device'])
    vm,vp,vpred,vclass=compute_metrics(val['y_true'],val['logits']); tm,tp,tpred,tclass=compute_metrics(test['y_true'],test['logits'])
    pd.DataFrame([{'split':'val',**vm,'loss':val['loss']},{'split':'test',**tm,'loss':test['loss']}]).to_csv(metrics_dir/'final_metrics.csv',index=False)
    vclass.insert(0,'split','val'); tclass.insert(0,'split','test'); pd.concat([vclass,tclass],ignore_index=True).to_csv(metrics_dir/'per_class_metrics.csv',index=False)
    save_confusion(val['y_true'],vpred,metrics_dir,'val'); save_confusion(test['y_true'],tpred,metrics_dir,'test'); save_curves(val['y_true'],vp,metrics_dir,'val'); save_curves(test['y_true'],tp,metrics_dir,'test')
    df=meta_test.iloc[test['idx']].copy().reset_index(drop=True); df['true_class_index']=test['y_true']; df['pred_class_index']=tpred
    for i,n in enumerate(MAIN_LABELS): df[f'prob_{n}']=tp[:,i]
    df.to_csv(pred_dir/'test_predictions.csv',index=False)
    torch.save({'model_state_dict':model.state_dict(),'config':CONFIG,'model_name':model_name,'label_mapping':{'main_label_display':MAIN_LABEL_DISPLAY,'main_label_order':MAIN_LABELS}},transfer_dir/f'{model_name}_full_model.pt')
    summary.append({'model_name':model_name,**{f'test_{k}':v for k,v in tm.items() if isinstance(v,(float,int,np.floating))},'output_dir':str(model_dir),'checkpoint_path':str(ckpt_dir/'best_macro_f1.pt')})
summary_df=pd.DataFrame(summary).sort_values('test_macro_f1',ascending=False); summary_df.to_csv(OUTPUT_DIR/'metrics_summary.csv',index=False); display(summary_df)
